# 02 - Baseline Model - Telco Customer Churn

## Introduction

**Contexte.** TelcoWave, opérateur télécom (mobile + fibre) en Europe, veut réduire le churn
au prochain trimestre via un programme de rétention ciblé sous **budget marketing limité**.
L'objectif final du projet est un modèle de **scoring** estimant la probabilité de churn de
chaque client, pour prioriser les actions sur les plus à risque.

**Ce notebook (Étape 2 — Baseline).** Le notebook 01 a figé la connaissance du dataset et le
protocole d'évaluation. On construit ici le **premier modèle opérationnel** : une `Pipeline`
scikit-learn complète (nettoyage, encodage, scaling), un `DummyClassifier` comme plancher
de référence, puis une `LogisticRegression` comme baseline interprétable. Toutes les
transformations sont apprises **uniquement sur le train** pour garantir l'absence de fuite.
Le jeu de test reste intouché jusqu'à la Phase 3.

**Rappel données.** 7 043 clients × 21 colonnes. Cible : `Churn` (Yes/No), taux global
**26,54 %** (déséquilibre modéré). Drivers principaux identifiés en EDA : `Contract`
(Cramér's V ≈ 0.41), `OnlineSecurity` (0.35), `TechSupport` (0.34), `tenure` (Pearson
signé −0.35). Variables non-informatives exclues : `gender`, `PhoneService`.

## Sommaire

1. **Chargement et reconstitution des splits** — rechargement du CSV brut, jointure sur les IDs persistés, assertions de non-chevauchement.
2. **Nettoyage pré-Pipeline et features dérivées** — conversion `TotalCharges`, drapeau `totalcharges_is_missing`, feature `nb_services`.
3. **Construction de la Pipeline** — `ColumnTransformer` (imputation + scaling numérique, OHE catégoriel), séparation X/y.
4. **Modèles baseline** — `DummyClassifier(strategy="stratified")` comme plancher, `LogisticRegression(class_weight="balanced")` comme référence interprétable.
5. **Évaluation sur la validation** — ROC-AUC, PR-AUC, recall@10 %, precision@10 %, matrice de confusion, F1 au seuil 0.5.
6. **Sérialisation et scoring** — `joblib.dump` du pipeline, CSV `(customerID, proba_churn)` sur le test.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
)

from src.config import DATA_RAW, DATA_PROCESSED, MODELS_DIR, RANDOM_STATE, TARGET

## Section 1: Chargement et reconstitution des splits

In [2]:
# Chargement du CSV brut
df = pd.read_csv(DATA_RAW)
print(f"Brut chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# Chargement des listes d'IDs
ids_train = pd.read_csv(DATA_PROCESSED / "split_train.csv")["customerID"]
ids_valid  = pd.read_csv(DATA_PROCESSED / "split_valid.csv")["customerID"]
ids_test   = pd.read_csv(DATA_PROCESSED / "split_test.csv")["customerID"]

# Reconstitution des trois ensembles
train_df = df[df["customerID"].isin(ids_train)].copy()
valid_df  = df[df["customerID"].isin(ids_valid)].copy()
test_df   = df[df["customerID"].isin(ids_test)].copy()

# Assertions anti-chevauchement
assert set(ids_train) & set(ids_valid) == set(), "Chevauchement train/valid !"
assert set(ids_train) & set(ids_test)  == set(), "Chevauchement train/test !"
assert set(ids_valid) & set(ids_test)  == set(), "Chevauchement valid/test !"
assert len(train_df) + len(valid_df) + len(test_df) == len(df), "Perte de lignes !"

print(f"\nTrain : {len(train_df):,} | Valid : {len(valid_df):,} | Test : {len(test_df):,}")
print(f"Total reconstitué : {len(train_df)+len(valid_df)+len(test_df):,} (attendu {len(df):,})")

Brut chargé : 7,043 lignes × 21 colonnes

Train : 4,225 | Valid : 1,409 | Test : 1,409
Total reconstitué : 7,043 (attendu 7,043)


## Section 2: Nettoyage pré-Pipeline et features dérivées

Deux opérations effectuées **avant** la Pipeline, sur les trois ensembles :

- `TotalCharges` : converti en numérique (11 espaces blancs -> `NaN`). Le drapeau
  `totalcharges_is_missing` est créé **avant** l'imputation, sinon il serait toujours `False`.
- `nb_services` : compte des services actifs par client (hors `No`, `No phone service`,
  `No internet service`). Relation en cloche avec le churn identifiée en EDA non capturable
  par Pearson. Calculé ici comme feature dérivée, pas dans la Pipeline.
- `churn_bin` : encodage binaire de la cible (Yes -> 1, No -> 0).

In [3]:
# Colonnes de services à compter
SERVICE_COLS = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
NO_SERVICE_VALUES = {"No", "No phone service", "No internet service"}

def prepare(df):
    df = df.copy()
    # Conversion TotalCharges : espaces blancs → NaN
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    # Drapeau créé avant toute imputation
    df["totalcharges_is_missing"] = df["TotalCharges"].isna().astype(int)
    # Feature dérivée : nombre de services actifs
    df["nb_services"] = df[SERVICE_COLS].apply(
        lambda row: sum(v not in NO_SERVICE_VALUES for v in row), axis=1
    )
    # Encodage binaire de la cible
    df["churn_bin"] = (df[TARGET] == "Yes").astype(int)
    return df

train_df = prepare(train_df)
valid_df  = prepare(valid_df)
test_df   = prepare(test_df)

print("TotalCharges NaN — train :", train_df["TotalCharges"].isna().sum())
print("nb_services — train (5 premiers) :", train_df["nb_services"].head().tolist())
print("churn_bin — train (taux) :", train_df["churn_bin"].mean().round(4))

TotalCharges NaN — train : 7
nb_services — train (5 premiers) : [1, 2, 0, 4, 1]
churn_bin — train (taux) : 0.2653
